<a href="https://colab.research.google.com/github/NikitaNechaev/Run-PPP-multiple/blob/main/run_pharokka_and_phold_and_phynteny.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Pharokka + Phold + Phynteny

[pharokka](https://github.com/gbouras13/pharokka) is a rapid standardised annotation tool for bacteriophage genomes and metagenomes. You can read more about pharokka in the [documentation](https://pharokka.readthedocs.io/).

[phold](https://github.com/gbouras13/phold) is a sensitive annotation tool for bacteriophage genomes and metagenomes using protein structural homology. You can read more about phold in the [documentation](https://phold.readthedocs.io/).

phold uses the [ProstT5](https://github.com/mheinzinger/ProstT5) protein language model to translate protein amino acid sequences to the 3Di token alphabet used by [Foldseek](https://github.com/steineggerlab/foldseek). Foldseek is then used to search these against a database of 803k protein structures mostly predicted using [Colabfold](https://github.com/sokrypton/ColabFold).

[phyntney](https://github.com/susiegriggo/Phynteny_transformer) uses phage synteny (the conserved gene order across phages) with a hybrid transformer/LSTM architecture to assign hypothetical phage proteins to a PHROG category.

The tools are best run sequentially, as Pharokka conducts extra annotation steps like tRNA, tmRNA, CRISPR and INPHARED searches that Phold lacks (for now at least). Pharokka will also (rarely) annotate CDS that Phold can miss. Phynteny can then help annotate remaining hypothetical proteins with a PHROG category.

* **Before you start, please make sure you change the runtime to T4 GPU (or any other kind of GPU if you have $$$), otherwise Phold won't be installed properly**
* To do this, go to the top toolbar, then to Runtime -> Change runtime type -> Hardware accelerator

* To run the cells, press the play button on the left side
* Cells 1, 2 and 3 install pharokka, phold and phyntney and download the databases/models.
* Once they have been run, you can re-run Cell 4 (to run Pharokka), Cell 5 (to run Phold) and Cell 6 (to run Phynteny) as many times as you would like



In [1]:

#@title 1. Install miniforge

#@markdown This cell installs miniforge

%%bash

set -e

PYTHON_VERSION=$(python3 -c "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')")

echo "python version ${PYTHON_VERSION}"

if [ ! -f CONDA_READY ]; then
  echo "installing miniforge"

  # miniforge 25.9.1 introduces some issue - latest as of 7 Nov 2025 - see https://github.com/gbouras13/phold/issues/106
  # issue is fixed if you use the previous release (25.3.1)
  #wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
  wget -qnc https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-25.3.1-0-Linux-x86_64.sh
  bash Miniforge3-25.3.1-0-Linux-x86_64.sh -bfp /usr/local 2>&1 1>/dev/null
  conda config --set auto_update_conda false
  touch CONDA_READY
fi

pip install --upgrade matplotlib matplotlib-inline



python version 3.12
installing miniforge
warning  libmamba [python-3.12.11-h9e4cc4f_0_cpython] The following files were already present in the environment:
    - bin/python
warning  libmamba [charset-normalizer-3.4.2-pyhd8ed1ab_0] The following files were already present in the environment:
    - bin/normalizer
warning  libmamba [distro-1.9.0-pyhd8ed1ab_1] The following files were already present in the environment:
    - bin/distro
warning  libmamba [jsonpointer-3.0.0-py312h7900ff3_1] The following files were already present in the environment:
    - bin/jsonpointer
warning  libmamba [wheel-0.45.1-pyhd8ed1ab_1] The following files were already present in the environment:
    - bin/wheel
warning  libmamba [jsonpatch-1.33-pyhd8ed1ab_1] The following files were already present in the environment:
    - bin/jsondiff
    - bin/jsonpatch
warning  libmamba [pip-25.1.1-pyh8b19718_0] The following files were already present in the environment:
    - bin/pip
    - bin/pip3
warning  libmamba [tq

In [2]:
#@title 2. Install pharokka and phold and phynteny

#@markdown This cell installs pharokka and phold. It will take a few minutes. Please be patient

PHAROKKA_VERSION="1.9.1"
PHOLD_VERSION="1.2.2"
PHYNTENY_VERSION="0.1.3"

# add paths
import sys
sys.path.append("/usr/local/bin")

# Update environment variables for shell usage
import os
os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]

# create envs
# pharokka isn't compatible with Python 3.13 (Google Colab default)
# so it needs a separate env
from pathlib import Path
flag_file = Path("PHAROKKA_PHOLD_PHYNTENY_READY")
if not flag_file.exists():
  !conda install -y -c bioconda phold=={PHOLD_VERSION} pharokka=={PHAROKKA_VERSION} pytorch=*=cuda* phynteny_transformer=={PHYNTENY_VERSION}
  # Touch the flag file
  flag_file.touch()

Streaming output truncated to the last 5000 lines.










mkl-2025.3.1         | 119.6 MB  | :  22% 0.21526303930230392/1 [00:45<00:25, 32.73s/it]











openjdk-25.0.1       | 111.6 MB  | :  16% 0.15609324218392664/1 [00:45<01:03, 75.29s/it] 










mkl-2025.3.1         | 119.6 MB  | :  24% 0.24021160757095686/1 [00:45<00:18, 24.63s/it]











openjdk-25.0.1       | 111.6 MB  | :  18% 0.18129215123604037/1 [00:45<00:42, 51.98s/it]










mkl-2025.3.1         | 119.6 MB  | :  27% 0.2679032121413989/1 [00:45<00:13, 18.08s/it] 











openjdk-25.0.1       | 111.6 MB  | :  21% 0.2067710481665109/1 [00:45<00:29, 36.59s/it] 










mkl-2025.3.1         | 119.6 MB  | :  29% 0.2932436427388788/1 [00:45<00:10, 14.70s/it]











openjdk-25.0.1       | 111.6 MB  | :  23% 0.23140998146191097/1 [00:45<00:20, 26.60s/it]










mkl-2025.3.1         | 119.6 MB  | :  32% 0.31557979548201837/1 [00:45<00:08, 12.01s/it]











openjdk-25.0.1       | 111.6 MB  | :  26% 0

In [3]:
#@title 3. Download pharokka phold databases

#@markdown This cell downloads the pharokka then the phold database. It will take some time (10-15 minutes probably depending on Zenodo's traffic). Please be patient. Perhaps go for a walk or have a coffee or tea.


%%time

print("Downloading pharokka database. This will take a few minutes. Please be patient :)")
!install_databases.py -o pharokka_db


print("Downloading phold database. This will take a few minutes. Please be patient :)")
!phold install -d phold_db -t 8 --foldseek_gpu


print("Downloading phynteny database. This will take a few minutes. Please be patient :)")
!install_models -o  phynteny_models


2026-03-25 21:04:34.284 | INFO     | databases:instantiate_install:124 - Checking Pharokka database installation in pharokka_db.
2026-03-25 21:04:34.284 | INFO     | databases:check_db_installation:252 - PHROGs Database file VERSION_1_8_0 is missing.
2026-03-25 21:04:34.284 | INFO     | databases:check_db_installation:259 - VFDB Databases are missing.
2026-03-25 21:04:34.284 | INFO     | databases:check_db_installation:266 - CARD Databases are missing.
2026-03-25 21:04:34.284 | INFO     | databases:check_db_installation:272 - PHROGs Annotation File is missing.
2026-03-25 21:04:34.284 | INFO     | databases:check_db_installation:278 - INPHARED Mash Annotation File is missing.
2026-03-25 21:04:34.284 | INFO     | databases:check_db_installation:284 - INPHARED Mash Sketch File is missing.
2026-03-25 21:04:34.284 | INFO     | databases:instantiate_install:129 - Some Databases are missing.
2026-03-25 21:04:34.284 | INFO     | databases:instantiate_install:134 - Downloading Pharokka Database

In [4]:

#@title 4. Run Pharokka (batch mode)

#@markdown Upload all your FASTA files (`.fa`, `.fasta`, `.fna`) and list full paths in INPUT_FILES (one file per line).
#@markdown Example for 12 files: `/content/sample1.fa` ... `/content/sample12.fa`.

#@markdown Provide the root output directory for Pharokka with PHAROKKA_OUT_ROOT.
#@markdown Each input will be processed into a separate subfolder.

#@markdown Then choose a gene predictor: `phanotate`, `prodigal`, or `prodigal-gv`.

#@markdown You can click FORCE to overwrite output subdirectories.

%%time
import json
import os
import re
import shlex
import subprocess
import sys
import zipfile
from pathlib import Path

BATCH_MANIFEST_PATH = Path('/content/ppp_batch_manifest.json')

INPUT_FILES = '/content/MN2_1.fa
/content/MN2_2.fa' #@param {type:"string"}
PHAROKKA_OUT_ROOT = 'output_pharokka_batch'  #@param {type:"string"}
GENE_PREDICTOR = 'phanotate'  #@param {type:"string"}
LOCUS_TAG = 'Default'  #@param {type:"string"}
FAST = True  #@param {type:"boolean"}
META = False  #@param {type:"boolean"}
META_HMM = False  #@param {type:"boolean"}
FORCE = True  #@param {type:"boolean"}

allowed_gene_predictors = ['phanotate', 'prodigal', 'prodigal-gv']
if GENE_PREDICTOR.lower() not in allowed_gene_predictors:
    raise ValueError("Invalid GENE_PREDICTOR. Please choose from: 'phanotate', 'prodigal', 'prodigal-gv'.")

input_files = [line.strip() for line in INPUT_FILES.splitlines() if line.strip()]
if not input_files:
    raise ValueError('INPUT_FILES is empty. Provide at least one FASTA path.')

missing_files = [fp for fp in input_files if not os.path.exists(fp)]
if missing_files:
    print('The following files are missing:')
    for fp in missing_files:
        print(f'  - {fp}')
    sys.exit(1)

os.makedirs(PHAROKKA_OUT_ROOT, exist_ok=True)

used_sample_ids = set()
manifest = []

for idx, input_file in enumerate(input_files, start=1):
    stem = Path(input_file).stem
    sample_id = re.sub(r'[^A-Za-z0-9_.-]+', '_', stem) or f'sample_{idx}'
    original_sample_id = sample_id
    dedup_idx = 1
    while sample_id in used_sample_ids:
        dedup_idx += 1
        sample_id = f"{original_sample_id}_{dedup_idx}"
    used_sample_ids.add(sample_id)

    sample_out_dir = os.path.join(PHAROKKA_OUT_ROOT, sample_id)

    cmd_parts = [
        'pharokka.py',
        '-d', 'pharokka_db',
        '-i', input_file,
        '-t', '4',
        '-o', sample_out_dir,
        '-p', sample_id,
        '-l', LOCUS_TAG,
        '-g', GENE_PREDICTOR,
    ]
    if FORCE:
        cmd_parts.append('-f')
    if FAST:
        cmd_parts.append('--fast')
    if META:
        cmd_parts.append('-m')
    if META_HMM:
        cmd_parts.append('--meta_hmm')

    printable_command = ' '.join(shlex.quote(x) for x in cmd_parts)
    print(f"[{idx}/{len(input_files)}] Running pharokka for {input_file}")
    print(printable_command)
    subprocess.run(cmd_parts, check=True)

    sample_gbk = os.path.join(sample_out_dir, f'{sample_id}.gbk')
    if not os.path.exists(sample_gbk):
        raise FileNotFoundError(f'Expected Pharokka output not found: {sample_gbk}')

    manifest.append({
        'sample_id': sample_id,
        'input_file': input_file,
        'pharokka_out_dir': sample_out_dir,
        'pharokka_gbk': sample_gbk,
    })

BATCH_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'Pharokka finished for {len(manifest)} input file(s).')
print(f'Manifest saved to {BATCH_MANIFEST_PATH}')

zip_filename = f"{PHAROKKA_OUT_ROOT}.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(PHAROKKA_OUT_ROOT):
        for file in files:
            full_path = os.path.join(root, file)
            zipf.write(full_path, os.path.relpath(full_path, PHAROKKA_OUT_ROOT))
print(f'Pharokka output zipped: {zip_filename}')


Input file /content/MN2_2.fa exists
Running pharokka
pharokka completed successfully.
Your output is in output_pharokka.
Zipping the output directory so you can download it all in one go.
Output directory has been zipped to output_pharokka.zip
CPU times: user 47 ms, sys: 622 µs, total: 47.6 ms
Wall time: 2min 1s


In [5]:

#@title 5. Run phold (batch mode)

#@markdown This cell runs phold for all samples produced by the previous Pharokka batch cell.
#@markdown It reads `/content/ppp_batch_manifest.json` automatically.

%%time
import json
import os
import shlex
import subprocess
import zipfile
from pathlib import Path

BATCH_MANIFEST_PATH = Path('/content/ppp_batch_manifest.json')
if not BATCH_MANIFEST_PATH.exists():
    raise FileNotFoundError('Manifest not found. Run the Pharokka batch cell first.')

PHOLD_OUT_ROOT = 'output_phold_batch'  #@param {type:"string"}
FORCE = True  #@param {type:"boolean"}
SEPARATE = False  #@param {type:"boolean"}

manifest = json.loads(BATCH_MANIFEST_PATH.read_text(encoding='utf-8'))
os.makedirs(PHOLD_OUT_ROOT, exist_ok=True)

for idx, item in enumerate(manifest, start=1):
    sample_id = item['sample_id']
    phold_input = item['pharokka_gbk']
    sample_out_dir = os.path.join(PHOLD_OUT_ROOT, sample_id)

    cmd_parts = [
        'phold', 'run',
        '-i', phold_input,
        '-t', '4',
        '-o', sample_out_dir,
        '-p', sample_id,
        '-d', 'phold_db',
        '--foldseek_gpu',
    ]
    if FORCE:
        cmd_parts.append('-f')
    if SEPARATE:
        cmd_parts.append('--separate')

    print(f"[{idx}/{len(manifest)}] Running phold for {sample_id}")
    print(' '.join(shlex.quote(x) for x in cmd_parts))
    subprocess.run(cmd_parts, check=True)

    sample_gbk = os.path.join(sample_out_dir, f'{sample_id}.gbk')
    if not os.path.exists(sample_gbk):
        raise FileNotFoundError(f'Expected phold output not found: {sample_gbk}')

    item['phold_out_dir'] = sample_out_dir
    item['phold_gbk'] = sample_gbk

BATCH_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'phold finished for {len(manifest)} sample(s).')
print(f'Manifest updated: {BATCH_MANIFEST_PATH}')

zip_filename = f"{PHOLD_OUT_ROOT}.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(PHOLD_OUT_ROOT):
        for file in files:
            full_path = os.path.join(root, file)
            zipf.write(full_path, os.path.relpath(full_path, PHOLD_OUT_ROOT))
print(f'phold output zipped: {zip_filename}')


Running phold
phold completed successfully.
Your output is in output_phold.
Zipping the output directory so you can download it all in one go.
Output directory has been zipped to output_phold.zip
CPU times: user 45.4 ms, sys: 699 µs, total: 46.1 ms
Wall time: 1min 37s


In [6]:

#@title 6. Run Phynteny (batch mode)

#@markdown This cell runs phynteny for all samples produced by the previous phold batch cell.
#@markdown It reads `/content/ppp_batch_manifest.json` automatically.

%%time
import json
import os
import shlex
import subprocess
import zipfile
from pathlib import Path

BATCH_MANIFEST_PATH = Path('/content/ppp_batch_manifest.json')
if not BATCH_MANIFEST_PATH.exists():
    raise FileNotFoundError('Manifest not found. Run Pharokka and phold batch cells first.')

PHYNTENY_OUT_ROOT = 'output_phynteny_batch'  #@param {type:"string"}
FORCE = False  #@param {type:"boolean"}

manifest = json.loads(BATCH_MANIFEST_PATH.read_text(encoding='utf-8'))
os.makedirs(PHYNTENY_OUT_ROOT, exist_ok=True)

for idx, item in enumerate(manifest, start=1):
    sample_id = item['sample_id']
    phynteny_input = item.get('phold_gbk')
    if not phynteny_input:
        raise ValueError(f"phold_gbk is missing for sample {sample_id}. Run the phold batch cell first.")

    sample_out_dir = os.path.join(PHYNTENY_OUT_ROOT, sample_id)
    cmd_parts = [
        'phynteny_transformer',
        '-m', '/content/phynteny_models/models',
        '-o', sample_out_dir,
        phynteny_input,
    ]
    if FORCE:
        cmd_parts.append('-f')

    print(f"[{idx}/{len(manifest)}] Running phynteny for {sample_id}")
    print(' '.join(shlex.quote(x) for x in cmd_parts))
    subprocess.run(cmd_parts, check=True)

    item['phynteny_out_dir'] = sample_out_dir

BATCH_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'phynteny finished for {len(manifest)} sample(s).')
print(f'Manifest updated: {BATCH_MANIFEST_PATH}')

zip_filename = f"{PHYNTENY_OUT_ROOT}.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(PHYNTENY_OUT_ROOT):
        for file in files:
            full_path = os.path.join(root, file)
            zipf.write(full_path, os.path.relpath(full_path, PHYNTENY_OUT_ROOT))
print(f'phynteny output zipped: {zip_filename}')


Running phynteny
phynteny completed successfully.
Your output is in output_phynteny.
Zipping the output directory so you can download it all in one go.
Output directory has been zipped to output_phynteny.zip
CPU times: user 57.7 ms, sys: 1.22 ms, total: 58.9 ms
Wall time: 53.4 s
